# PCA Analysis of Meteorite Landings

This notebook demonstrates principal component analysis (PCA) on the meteorite landings dataset.  
We will:

1. Load and inspect the data  
2. Select and clean numeric features  
3. Standardize features  
4. Fit PCA and examine loadings  
5. Plot a scree plot  
6. Visualize PC1 vs PC2  
7. Color the PC1–PC2 scatter by meteorite class  


In [ ]:
# Cell 2: Imports & Setup
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

%matplotlib inline


In [ ]:
# Cell 3: Load & Inspect Data
csv_path = '/Users/pbat/Projects/cmor438/data/meteorite-landings.csv'
df = pd.read_csv(csv_path)

# show first rows, shape, and columns
df.head(), df.shape, df.columns.tolist()


In [ ]:
# Cell 4: Select & Clean Numeric Features
numeric_cols = ['mass', 'reclat', 'reclong']
df_num = df[numeric_cols].dropna()

# how many rows remain?
df_num.shape


In [ ]:
# Cell 5: Standardize & Fit PCA, then compute loadings
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_num)

pca = PCA()
X_pca = pca.fit_transform(X_scaled)

# compute loadings (feature weights)
loadings = pd.DataFrame(
    pca.components_.T,
    index=numeric_cols,
    columns=[f'PC{i}' for i in range(1, pca.n_components_+1)]
)
print("Feature loadings for each PC:")
display(loadings)

print("\nExplained variance ratio per PC:")
for i, ratio in enumerate(pca.explained_variance_ratio_, start=1):
    print(f"  PC{i}: {ratio:.4f}")


In [ ]:
# Cell 6: Plot
plt.figure()
plt.plot(
    range(1, len(pca.explained_variance_ratio_)+1),
    pca.explained_variance_ratio_,
    marker='o'
)
plt.title('Scree Plot')
plt.xlabel('Principal Component')
plt.ylabel('Variance Explained')
plt.xticks(range(1, len(pca.explained_variance_ratio_)+1))
plt.show()


In [ ]:
# Cell 7: PC1 vs PC2 Scatter
var1, var2 = pca.explained_variance_ratio_[:2]

plt.figure()
plt.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.6)
plt.title('PC1 vs PC2')
plt.xlabel(f'PC1 ({var1:.1%} var)')
plt.ylabel(f'PC2 ({var2:.1%} var)')
plt.show()


In [ ]:
# Cell 8: PC1 vs PC2 Colored by Meteorite Class
df_pcs = pd.DataFrame(X_pca[:, :2], columns=['PC1', 'PC2'], index=df_num.index)
df_pcs['recclass'] = df.loc[df_num.index, 'recclass']

plt.figure()
for cls in df_pcs['recclass'].unique():
    subset = df_pcs[df_pcs['recclass'] == cls]
    plt.scatter(subset['PC1'], subset['PC2'], label=cls, s=10, alpha=0.6)

plt.legend(markerscale=2, fontsize='small', loc='best')
plt.title('PC1 vs PC2 by Meteorite Class')
plt.xlabel(f'PC1 ({var1:.1%} var)')
plt.ylabel(f'PC2 ({var2:.1%} var)')
plt.show()
